# Processing gene names..

**Alyona you do not have to run this**

In [1]:
import pandas as pd

## Reuniting XLOC and Platybrowser gene names

I read the platybrowser label_id x gene table.

In [2]:
df_platy = pd.read_csv('./source_data/platybrowser/genes.tsv', sep='\t')
platybrowser_genes = df_platy.columns.to_series()[1:] # 0 is label_id
platybrowser_genes = platybrowser_genes[~platybrowser_genes.str.contains(r'edu.*')] # remove EdU, this is a dye staining
platybrowser_genes.to_list()[1:10] # all in lower case

['allcr1', 'allcr2', 'anpra', 'anprb', 'ap2', 'arx', 'ascb', 'asci', 'ascii']

I read our HCR database information from https://docs.google.com/spreadsheets/d/1VZDk0m41o4KzXDyTPpNG5QUuF03KRNTruyKUQaXKzi8/edit?usp=sharing

In [3]:
df_hcr_probes = pd.read_csv('./source_data/6dpf neuron types - Prospr_db_copy.tsv', sep='\t')
columns_to_extract = ['ProsPR gene', 'Published name', 'pdumV2-1', 'Platybrowser name']
df_hcr_probes = df_hcr_probes[columns_to_extract]
df_hcr_probes['Platybrowser name'] = df_hcr_probes['Platybrowser name'].str.lower() # capitalization 
df_hcr_probes.head()

,ProsPR gene,Published name,pdumV2-1,Platybrowser name
0,Glt1,Glt1,XLOC_001410,glt1
1,AChE,AChE,"XLOC_027437,XLOC_032912,XLOC_032910",ache
2,ANPRA,NaN,XLOC_025950,anpra
3,ANPRB,NaN,NaN,anprab
4,AP2,ap2,XLOC_002429,ap2


I am fixing a few capitalization issues and typos. 

* We have `calexcitin` and `junctophilin` with numbers appended (2 and 1) in Platybrowser.
* I have `anpr-a` and `anpr-b` (official symbols are NPR1 and NPR2). anpr-a is XLOC_025950. anpr-b has a hit in the genome but no XLOC, genome assembly issue... I drop the b form for now.

In [4]:
df_hcr_probes['Platybrowser name'] = df_hcr_probes['Platybrowser name'].str.replace('calexcitin', 'calexcitin2')
df_hcr_probes['Platybrowser name'] = df_hcr_probes['Platybrowser name'].str.replace('junctophilin', 'junctophilin1')
df_hcr_probes =df_hcr_probes[df_hcr_probes['Platybrowser name'] != 'anprab']

I ignore whatever is left.

In [5]:
set(platybrowser_genes).symmetric_difference(set(df_hcr_probes['Platybrowser name']))

{'anprb', nan}

In [6]:
df_hcr_probes = df_hcr_probes[df_hcr_probes['Platybrowser name'].isin(platybrowser_genes)]
df_hcr_probes

,ProsPR gene,Published name,pdumV2-1,Platybrowser name
0,Glt1,Glt1,XLOC_001410,glt1
1,AChE,AChE,"XLOC_027437,XLOC_032912,XLOC_032910",ache
2,ANPRA,NaN,XLOC_025950,anpra
4,AP2,ap2,XLOC_002429,ap2
5,ASCb,ASCb,XLOC_008377,ascb
...,...,...,...,...
197,brn124,NaN,XLOC_005393,brn124
198,nompc3,NaN,XLOC_002090,nompc3
199,pkd1,PKD1-1,XLOC_027897,pkd1
200,pkd2,PKD2-1,XLOC_026080,pkd2


I will make a few corrections and drop most of the problematic stuff.

In [7]:
df_hcr_probes.loc[df_hcr_probes['Platybrowser name'] == 'ache','pdumV2-1'] = 'XLOC_027437'
df_hcr_probes.loc[df_hcr_probes['Platybrowser name'] == 'gnrhr','pdumV2-1'] = 'XLOC_003285'
df_hcr_probes.loc[df_hcr_probes['Platybrowser name'] == 'nov2','pdumV2-1'] = 'XLOC_033937'

remove_those_ids = [
    'gppcr21',  #   - gppcr21 because I don't like the reverse BLAST results i am seeing. HCR probe was weirdly chosen
    'glyt',     #   - glyt too many hits for now
    'gnrh',     #   - gnrh need to add it to the transcriptome and rerun
    'hnf6',     #   - hnf6 """
    'hr38',     #   - hr38 multi hit
    'loc5285ct',#   - hr38 multi hit
    'isl',      #   - isl need to add it to the genome/transcriptome and rerun
    'lhx2',     #   - lhx2 need to add it to the genome/transcriptome and rerun
    'munc22',   #   - munc22 need to add it to the genome/transcriptome and rerun
    'muncb',    #   - munc22 need to add it to the genome/transcriptome and rerun
    'nmdar',    #   - nmdar need to add it to the genome/transcriptome and rerun
    'pax6',     #   - pax6 need to add it to the genome/transcriptome and rerun
    'sox2',     #   - sox2 need to add it to the genome/transcriptome and rerun
    'soxb12',   #   - soxb2 need to add it to the genome/transcriptome and rerun
    "vacht",    #   - vacht will be dropped, not a big loss of information
    'catl',     #   - catl  Out of time to find this
    'gpb',      #   - gpb   """
    'pea3',
    'baiap',
    'cal2',
    'calmodulin',
    'p56',
    'mecom'
]
df_hcr_probes_filtered = df_hcr_probes[~df_hcr_probes['Platybrowser name'].str.lower().str.strip().isin(remove_those_ids)]


I need hyphens for Seurat, and I would like to add gene symbols.

In [8]:
df_hcr_probes_filtered  = df_hcr_probes_filtered.assign(XLOC = df_hcr_probes_filtered ['pdumV2-1'].str.replace("_", "-"))
df_hcr_probes_filtered  = df_hcr_probes_filtered.assign(XLOC_full = df_hcr_probes_filtered ['XLOC'] + " | " + df_hcr_probes_filtered['Platybrowser name'])
df_hcr_probes_filtered.to_csv("./prepped_data/genes_metadata.tsv", sep= '\t')
df_hcr_probes_filtered[:21]

,ProsPR gene,Published name,pdumV2-1,Platybrowser name,XLOC,XLOC_full
0,Glt1,Glt1,XLOC_001410,glt1,XLOC-001410,XLOC-001410 | glt1
1,AChE,AChE,XLOC_027437,ache,XLOC-027437,XLOC-027437 | ache
2,ANPRA,NaN,XLOC_025950,anpra,XLOC-025950,XLOC-025950 | anpra
4,AP2,ap2,XLOC_002429,ap2,XLOC-002429,XLOC-002429 | ap2
5,ASCb,ASCb,XLOC_008377,ascb,XLOC-008377,XLOC-008377 | ascb
6,AllCR1,NaN,XLOC_022100,allcr1,XLOC-022100,XLOC-022100 | allcr1
7,AllCR2,NaN,XLOC_025660,allcr2,XLOC-025660,XLOC-025660 | allcr2
8,Arx,Arx,XLOC-044659,arx,XLOC-044659,XLOC-044659 | arx
9,Ash,ASH,XLOC_004258,ash,XLOC-004258,XLOC-004258 | ash
11,BarH1,BarH1,XLOC_056544,barh1,XLOC-056544,XLOC-056544 | barh1


## Processing the platybrowser metadata

We keep the columns and rename them to the `XLOC_full` which will be easier to follow.

In [9]:
df_platy = df_platy[['label_id'] + df_hcr_probes_filtered['Platybrowser name'].to_list()]
# LUT for XLOC
rename_LUT_dict = pd.Series(
    df_hcr_probes_filtered['XLOC_full'].values,
    index=df_hcr_probes_filtered['Platybrowser name']
).to_dict()
df_platy = df_platy.rename(columns=rename_LUT_dict)
df_platy.to_csv("./prepped_data/platybrowser_counts.tsv", sep= '\t')
df_platy

,label_id,XLOC-001410 | glt1,XLOC-027437 | ache,XLOC-025950 | anpra,XLOC-002429 | ap2,XLOC-008377 | ascb,XLOC-022100 | allcr1,XLOC-025660 | allcr2,XLOC-044659 | arx,XLOC-004258 | ash,...,XLOC-012610 | six12,XLOC-059161 | six4,XLOC-052205 | tlx,XLOC-000131 | trk,XLOC-027840 | uncx,XLOC-005393 | brn124,XLOC-002090 | nompc3,XLOC-027897 | pkd1,XLOC-026080 | pkd2,XLOC-026293 | ato2
0,1.0,0.003589,0.006695,0.00434,0.0,0.003445,0.000092,0.002875,0.0,0.000556,...,0.042993,0.002724,0.000474,0.000066,0.110408,0.007642,0.000176,0.142584,0.033726,0.008004
1,2.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,3.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.052785,0.000000,0.000000,0.000000,0.000000,0.000000,0.543866,0.210047,0.000000
3,4.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.011179,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.002033,0.000000,0.591463,0.012195,0.000000
4,5.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32694,32695.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
32695,32696.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
32696,32697.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
32697,32698.0,0.000000,0.465969,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.004188,...,0.193717,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.728796,0.332984,0.000000
